In [1]:
import pandas as pd
import plotly.express as px
import geopandas as gpd
import plotly.graph_objects as go
import json
import numpy as np 

In [2]:
##### Données du dataframe patients 
df = pd.read_csv("H:/canc_air/data/data_octobre_2023/Pseudonymisation_provisoire_geocoded_spatial.csv", sep = ";")
df.drop('Unnamed: 0', axis=1, inplace=True)

In [3]:
## Transformation du crs du dataframe WGS84 vers Lambert93 
gdf = (gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.x, df.y)).set_crs(epsg=4326))#.to_crs(epsg=2154)
df_iris = gpd.read_file('H:/canc_air/data/zones_geographiques/iris/CONTOURS-IRIS.shp')


In [6]:
print(f"Nombre de patients ayant un code departement nul : {gdf.CODE_DEPT.isnull().sum()}")
gdf_no_na = gdf.dropna(subset=["CODE_DEPT"])

Nombre de patients ayant un code departement nul : 4618


In [4]:
def mise_en_forme_figure(gdf, df_spatiale, CODE_zone, path_file): 
    ## Nombre de patients par departement : 
    patient_counts = gdf[CODE_zone].value_counts().reset_index()
    patient_counts.columns = [CODE_zone, 'patient_count']

    if CODE_zone == 'CODE_EPCI':
        patient_counts[CODE_zone]= patient_counts[CODE_zone].astype(int)
        df_spatiale[CODE_zone]= df_spatiale[CODE_zone].astype(int)
    else : 
        patient_counts[CODE_zone]= patient_counts[CODE_zone].astype(str)
        df_spatiale[CODE_zone]= df_spatiale[CODE_zone].astype(str)

    ## Ajout des géométries au nouveau tableau des patients par dept : 
    spatial_with_patients = patient_counts.merge(df_spatiale, on=CODE_zone, how='right')

    spatial_with_patients['patient_count'] = spatial_with_patients['patient_count'].fillna(0)

    #spatial_with_patients = spatial_with_patients.dropna(subset=['patient_count']) 

    spatial_with_patients_simplified = spatial_with_patients[[CODE_zone, 'patient_count', 'geometry']]

    df_spatiale = df_spatiale.to_crs(epsg=4326)
    # Convert the department shapes to GeoJSON
    df_spatiale.to_file(path_file+".geojson", driver='GeoJSON')

    with open(path_file+".geojson") as f:
        geojson_spatiale = json.load(f)

    return spatial_with_patients_simplified, geojson_spatiale, patient_counts

In [7]:
iris_w_patient, iris_geojson, patient_counts = mise_en_forme_figure(gdf_no_na, df_iris,'CODE_IRIS', "H:/canc_air/data/zones_geographiques/iris/iris") 

In [19]:
contrasted_color_scale = [
    [0.0, '#313695'],  # dark blue
    [0.1, '#4575b4'],  # blue
    [0.2, '#74add1'],  # light blue
    [0.3, '#abd9e9'],  # lighter blue
    [0.4, '#e0f3f8'],  # very light blue
    [0.5, '#fee090'],  # light orange
    [0.6, '#fdae61'],  # orange
    [0.7, '#f46d43'],  # red-orange
    [0.8, '#d73027'],  # red
    [0.9, '#a50026'],  # dark red
    [1.0, '#67001f']   # darker red
]
contrasted_color_scale2 = [
    [0.0, 'grey'],
    [0.1, '#095086'],  
    [0.25, '#9bd2f2'],  
    [0.5, '#fffff'],  
    [0.75, '#eca6a6'], 
    [1, '#b00000']  
]

iris_w_patient_color = iris_w_patient.copy()

# Assign a color based on the patient count
iris_w_patient_color['color'] = iris_w_patient['patient_count']


# Update your figure creation code with the new color scale
fig = px.choropleth_mapbox(iris_w_patient_color,
                           geojson=iris_geojson,
                           locations="CODE_IRIS",
                           color="color",
                           featureidkey="properties.CODE_IRIS",
                           hover_name="CODE_IRIS",
                           mapbox_style="carto-positron",
                           zoom=5,
                           opacity=0.5,
                           color_continuous_scale='Plotly3',
                           center={"lat": 46.2276, "lon": 2.2137},
                           labels={'count': 'Nombre de personne'},
                           range_color=(0, iris_w_patient['patient_count'].max())  
                          )

# Update the layout with non-zero margins and a readable colorbar
fig.update_layout(margin={"r":30, "t":0, "l":0, "b":0},  # Updated margins
                  width=1200,
                  height=600,
                  coloraxis_colorbar=dict(
                      title='Patient Count',
                      tickvals=[0,10,20,30,40,50,60,70,80,90,100,110,120,130,140,150,160,170,180,190,200],# 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000],
                      ticktext=['0', '10', '20', '30', '40', '50', '60', '70', '80', '90', '100',"110","120","130","140","150","160","170","180","190","200"]
                  )
)


fig.show()